In [ ]:
{
    "cells": [
        {
            "attachments": {},
            "cell_type": "markdown",
            "id": "1decf919",
            "metadata": {
                "id": "1decf919"
            },
            "source": [
                "## Initial settings"
            ]
        },
        {
            "cell_type": "markdown",
            "id": "6bbcc60e",
            "metadata": {},
            "source": [
                "- weights\n",
                "- labels weights\n",
                "- augmentet pics (aug pics in batch)\n",
                "- slightly lower lr\n",
                "- delete new CART parameters\n",
                "- check metrics names"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 4,
            "id": "4817d3ed",
            "metadata": {
                "id": "4817d3ed"
            },
            "outputs": [],
            "source": [
                "import os\n",
                "import socket\n",
                "import random\n",
                "\n",
                "import numpy as np\n",
                "import pandas as pd\n",
                "from PIL import Image\n",
                "import torch\n",
                "import tqdm\n",
                "\n",
                "from william.dataset import InvoiceDataset, DatabaseConnection, DatabaseConfig\n",
                "from william.dataset.config.filters import InvoiceFilterConfig\n",
                "# from william.dataset.constants import (\n",
                "#     GROUND_TRUTH_MARKUP_NAME,\n",
                "    # GT_LINE_ITEMS_ROWS_MARKUP_NAME,\n",
                "#     INVOFOX_LINE_ITEMS_MARKUP,\n",
                "# )\n",
                "from model_training.dataset import WilliamModelDataset\n",
                "\n",
                "# from secret_constants import MLFLOW_USERNAME, MLFLOW_PASSWORD\n",
                "# from william.inference.constants import MLFLOW_TRACKING_URI\n",
                "\n",
                "os.environ[\"MLFLOW_TRACKING_USERNAME\"] = 'iana-ulianenko'\n",
                "os.environ[\"MLFLOW_TRACKING_PASSWORD\"] = 'NmtGIgRd'\n",
                "os.environ[\"MLFLOW_TRACKING_URI\"] = \"https://mlflow-cusxuj3jjq-nw.a.run.app\"\n",
                "MLFLOW_TRACKING_URI=\"https://mlflow-cusxuj3jjq-nw.a.run.app\"\n",
                "\n",
                "pd.set_option(\"max_colwidth\", 400)\n",
                "pd.set_option(\"display.max_columns\", 500)\n",
                "\n",
                "\n",
                "Image.MAX_IMAGE_PIXELS = 933120000\n",
                "\n",
                "MODEL_CHECKPOINTS_PATH = \"/home/iana/training/model_checkpoints\"\n",
                "TENSORBOARD_DIR = \"/home/iana/training/tb\"\n",
                "DATASET_PATH = \"/Users/yanaulyanenko/Documents/apron/dataset_prod_from_db_v4\"\n",
                "\n",
                "MODEL_PATH = (\n",
                "    \"/Users/yanaulyanenko/Documents/apron/apron-ml-models/saved_model_buyer_info\"\n",
                ")\n",
                "MODEL_IS_CHECKPOINT = False\n",
                "\n",
                "IS_CLASSIFICATION_MODE = True\n",
                "\n",
                "# GROUND_TRUTH_MARKUP_FOR_VAL = (\n",
                "#     william_markup  # LEDGER_MARKUP_NAME #'invofox_plus_veryfi_corrected'\n",
                "# )\n",
                "GROUND_TRUTH_MARKUP_FOR_VAL = \"ground_truth\"\n",
                "GROUND_TRUTH_MARKUP_NAME = \"ground_truth\"\n",
                "\n",
                "PROD_PREDICTION_NAME = \"william_prod\"\n",
                "\n",
                "GROUND_TRUTH_MARKUP = GROUND_TRUTH_MARKUP_NAME\n",
                "GROUND_TRUTH_MARKUP_FOR_VAL = GROUND_TRUTH_MARKUP_NAME\n",
                "GT_LINE_ITEMS_ROWS_MARKUP_NAME = \"gt_line_items_rows\"\n",
                "INVOFOX_LINE_ITEMS_MARKUP = \"invofox_line_items\"\n",
                "\n",
                "INVOFOX_LINE_ITEMS_ROWS_MARKUP = \"invofox_line_items_rows\"\n",
                "\n",
                "LEARNING_RATE = 2.5e-5\n",
                "EXPERIMENT_NAME = f\"capture-2216-train-new-dataset\"\n",
                "RUN_NAME = \"2-5-lr-011025\"\n",
                "\n",
                "HUGGINGFACE_MODEL_ID = \"SCUT-DLVCLab/lilt-roberta-en-base\"\n",
                "\n",
                "EXPERIMENT_CHECKPOINTS_PATH = os.path.join(MODEL_CHECKPOINTS_PATH, EXPERIMENT_NAME)\n",
                "os.makedirs(EXPERIMENT_CHECKPOINTS_PATH, exist_ok=True)\n",
                "os.makedirs(TENSORBOARD_DIR, exist_ok=True)\n",
                "\n",
                "torch.manual_seed(0)\n",
                "random.seed(0)\n",
                "np.random.seed(0)\n",
                "# torch.use_deterministic_algorithms(True)"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "id": "797ef4ed",
            "metadata": {},
            "outputs": [],
            "source": []
        },
        {
            "attachments": {},
            "cell_type": "markdown",
            "id": "pwPyC7-EwjQ0",
            "metadata": {
                "id": "pwPyC7-EwjQ0"
            },
            "source": [
                "## Dataset"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 5,
            "id": "a7c5e5e7",
            "metadata": {},
            "outputs": [],
            "source": [
                "from william.dataset import InvoiceDataset, DatabaseConnection, DatabaseConfig\n",
                "from william.dataset.config.filters import InvoiceFilterConfig"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 6,
            "id": "513760be",
            "metadata": {},
            "outputs": [],
            "source": [
                "# Create database configuration\n",
                "filter_config = InvoiceFilterConfig(\n",
                "            markup_name=[\"ground_truth\"],\n",
                "            # document_classes=[\"invoice\", \"receipt\"], # WORKS SLOW\n",
                "            # records_limit=40000\n",
                "        )\n",
                "\n",
                "\n",
                "\n",
                "config = DatabaseConfig(\n",
                "    host=\"localhost\",\n",
                "    port=5432,\n",
                "    database=\"apron-dwh\",\n",
                "    user=\"read-only\",\n",
                "    password=\"VcaRgbqesAZhK\",\n",
                "    # filter_config=filter_config,\n",
                "    pool_size=10\n",
                ")\n",
                "db_connection = DatabaseConnection(config)\n",
                "train_dataset = InvoiceDataset(\n",
                "            db_connection=db_connection,\n",
                "            split=\"train\",\n",
                "            filter_config=filter_config,\n",
                "            enable_cache=True,\n",
                "            # batch_size=16\n",
                "        )\n",
                "\n",
                "val_dataset = InvoiceDataset(db_connection, split='val', filter_config=filter_config, enable_cache=True,\n",
                "# batch_size=16\n",
                ")\n",
                "test_dataset = InvoiceDataset(\n",
                "    db_connection, split='test', filter_config=filter_config, enable_cache=True,\n",
                "    # batch_size=16\n",
                "    )"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 7,
            "id": "f302ea84",
            "metadata": {},
            "outputs": [
                {
                    "name": "stderr",
                    "output_type": "stream",
                    "text": [
                        "\u001b[32m2025-10-01 13:26:08\u001b[0m | \u001b[1mINFO    \u001b[0m | \u001b[36mwilliam.dataset.dataset.base\u001b[0m:\u001b[36m_load_cache_index\u001b[0m:\u001b[36m382\u001b[0m - \u001b[1mLoaded cache index: 8318 documents\u001b[0m\n",
                        "\u001b[32m2025-10-01 13:26:08\u001b[0m | \u001b[1mINFO    \u001b[0m | \u001b[36mwilliam.dataset.dataset.base\u001b[0m:\u001b[36m_ensure_cache_ready\u001b[0m:\u001b[36m407\u001b[0m - \u001b[1mUsing existing cache (age: 473.6 hours)\u001b[0m\n",
                        "\u001b[32m2025-10-01 13:26:08\u001b[0m | \u001b[1mINFO    \u001b[0m | \u001b[36mwilliam.dataset.dataset.base\u001b[0m:\u001b[36m_load_cache_index\u001b[0m:\u001b[36m382\u001b[0m - \u001b[1mLoaded cache index: 1373 documents\u001b[0m\n",
                        "\u001b[32m2025-10-01 13:26:08\u001b[0m | \u001b[1mINFO    \u001b[0m | \u001b[36mwilliam.dataset.dataset.base\u001b[0m:\u001b[36m_ensure_cache_ready\u001b[0m:\u001b[36m407\u001b[0m - \u001b[1mUsing existing cache (age: 472.9 hours)\u001b[0m\n",
                        "\u001b[32m2025-10-01 13:26:08\u001b[0m | \u001b[1mINFO    \u001b[0m | \u001b[36mwilliam.dataset.dataset.base\u001b[0m:\u001b[36m_load_cache_index\u001b[0m:\u001b[36m382\u001b[0m - \u001b[1mLoaded cache index: 2122 documents\u001b[0m\n",
                        "\u001b[32m2025-10-01 13:26:08\u001b[0m | \u001b[1mINFO    \u001b[0m | \u001b[36mwilliam.dataset.dataset.base\u001b[0m:\u001b[36m_ensure_cache_ready\u001b[0m:\u001b[36m407\u001b[0m - \u001b[1mUsing existing cache (age: 472.9 hours)\u001b[0m\n"
                    ]
                },
                {
                    "data": {
                        "text/plain": [
                            "(8318, 1373, 2122)"
                        ]
                    },
                    "execution_count": 7,
                    "metadata": {},
                    "output_type": "execute_result"
                }
            ],
            "source": [
                "from invoices_dataset.constants import GROUND_TRUTH_MARKUP_NAME\n",
                "# from model_training.dataset_filters import (\n",
                "#     has_ground_truth_markup,\n",
                "#     has_labelled_train_markup,\n",
                "#     has_new_val_markup,\n",
                "# )\n",
                "from william.inference.constants import INTERESTING_FIELDS\n",
                "from william.inference import WordLabel\n",
                "\n",
                "len(train_dataset), len(val_dataset), len(test_dataset)"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 8,
            "id": "aa993e9e",
            "metadata": {},
            "outputs": [
                {
                    "data": {
                        "text/plain": [
                            "PosixPath('/home/iana/.william_cache/invoice_datasets/invoice_test_bc6aa5561c6f.jsonl.gz')"
                        ]
                    },
                    "execution_count": 8,
                    "metadata": {},
                    "output_type": "execute_result"
                }
            ],
            "source": [
                "test_dataset.cache_data_file"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 9,
            "id": "2a58b0a0",
            "metadata": {},
            "outputs": [],
            "source": [
                "# ! cp /home/iana/.william_cache/invoice_datasets/invoice_val_e8c83fe7a90c.jsonl.gz /home/iana/"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 10,
            "id": "aa00d9a1",
            "metadata": {},
            "outputs": [],
            "source": [
                "# ! cp /home/iana/.william_cache/invoice_datasets/invoice_train_a557b5977a59.jsonl.gz /home/iana/"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 11,
            "id": "f62e62d7",
            "metadata": {},
            "outputs": [],
            "source": [
                "# ! cp /home/iana/.william_cache/invoice_datasets/invoice_test_bc6aa5561c6f.jsonl.gz /home/iana/"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "id": "aa9fa8c9",
            "metadata": {},
            "outputs": [],
            "source": []
        },
        {
            "cell_type": "code",
            "execution_count": 12,
            "id": "254aec1b",
            "metadata": {},
            "outputs": [],
            "source": [
                "# cnt = 0\n",
                "# for i in train_dataset:\n",
                "#     if i.document_classes != []:\n",
                "#         cnt +=1 \n",
                "# cnt\n"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 13,
            "id": "2a4e51ec",
            "metadata": {},
            "outputs": [],
            "source": [
                "from william.dataset.dataset.base import DocumentSerializer"
            ]
        },
        {
            "cell_type": "markdown",
            "id": "4ce9090c",
            "metadata": {},
            "source": [
                "### Fields extraction"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 14,
            "id": "fe70efea",
            "metadata": {},
            "outputs": [],
            "source": [
                "from concurrent.futures import ProcessPoolExecutor, as_completed, ThreadPoolExecutor\n",
                "from itertools import chain\n",
                "from tqdm import tqdm\n",
                "import os\n",
                "import json\n",
                "import gzip"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 15,
            "id": "4ba268da",
            "metadata": {},
            "outputs": [],
            "source": [
                "# val_markup  = val_dataset.read_serialized_dataset_by_page()\n",
                "# print(f\"Val size: {len(val_markup)}\")"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 16,
            "id": "51edac56",
            "metadata": {},
            "outputs": [
                {
                    "name": "stdout",
                    "output_type": "stream",
                    "text": [
                        "Val size: 1737\n"
                    ]
                }
            ],
            "source": [
                "val_markup  = val_dataset.read_serialized_dataset_by_page()\n",
                "print(f\"Val size: {len(val_markup)}\")"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 17,
            "id": "1e0ab660",
            "metadata": {},
            "outputs": [],
            "source": [
                "# test_markup  = test_dataset.read_serialized_dataset_by_page()\n",
                "# print(f\"Test size: {len(test_markup)}\")\n",
                "# train_markup = train_dataset.read_serialized_dataset_by_page()\n",
                "# print(f\"Train size: {len(train_markup)}\")"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 18,
            "id": "982137cb",
            "metadata": {},
            "outputs": [
                {
                    "name": "stdout",
                    "output_type": "stream",
                    "text": [
                        "Test size: 2693\n",
                        "Train size: 11348\n"
                    ]
                }
            ],
            "source": [
                "test_markup  = test_dataset.read_serialized_dataset_by_page()\n",
                "print(f\"Test size: {len(test_markup)}\")\n",
                "train_markup = train_dataset.read_serialized_dataset_by_page()\n",
                "print(f\"Train size: {len(train_markup)}\")"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 19,
            "id": "c9a3c3c9",
            "metadata": {},
            "outputs": [],
            "source": [
                "# def _process_one_doc(doc):\n",
                "#     s = DocumentSerializer.serialize_document(doc)\n",
                "#     doc_classes = s['document_classes']\n",
                "\n",
                "#     items = []\n",
                "#     for p in s['pages']:\n",
                "#         x = dict(p)\n",
                "#         x[\"bounding_boxes\"]      = p[\"total_bboxes\"]\n",
                "#         x[\"image_path\"]          = p['metadata']['image_path']\n",
                "#         x[\"classification_data\"] = doc_classes\n",
                "#         items.append(x)\n",
                "#     return items\n",
                "\n",
                "# def build_split_threads(data, workers=32):\n",
                "#     # data = list(dataset)\n",
                "#     results = []\n",
                "#     with ThreadPoolExecutor(max_workers=workers) as ex:\n",
                "#         futures = [ex.submit(_process_one_doc, d) for d in data]\n",
                "#         for fut in tqdm(as_completed(futures), total=len(futures), desc=\"Docs\", unit=\"doc\"):\n",
                "#             try:\n",
                "#                 results.append(fut.result())\n",
                "#             except Exception as e:\n",
                "#                 print(f\"Problem with doc: {e}\")\n",
                "#     return list(chain.from_iterable(results))\n",
                "\n",
                "\n",
                "# val_markup   = build_split_threads(val_dataset)\n",
                "# print(f\"Val size: {len(val_markup)}\")\n",
                "# test_markup  = build_split_threads(test_dataset)\n",
                "# print(f\"Test size: {len(test_markup)}\")\n",
                "# train_markup = build_split_threads(train_dataset)\n",
                "# print(f\"Train size: {len(train_markup)}\")"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 20,
            "id": "96705422",
            "metadata": {},
            "outputs": [],
            "source": [
                "# from tqdm import tqdm\n",
                "\n",
                "# def iter_doc_markup(doc):\n",
                "#     s = DocumentSerializer.serialize_document(doc)  # 1 раз на документ\n",
                "#     doc_classes = s['document_classes']\n",
                "\n",
                "#     for p in s['pages']:\n",
                "#         x = dict(p)\n",
                "#         x[\"bounding_boxes\"]     = p[\"total_bboxes\"]\n",
                "#         x[\"image_path\"]         = p['metadata']['image_path']\n",
                "#         x[\"classification_data\"] = doc_classes\n",
                "#         yield x\n",
                "\n",
                "# def build_split(dataset):\n",
                "#     out = []\n",
                "#     for doc in tqdm(dataset):\n",
                "#         try:\n",
                "#             out.extend(iter_doc_markup(doc))\n",
                "#         except Exception as e:\n",
                "#             print(f\"Problem with doc: {e}\")\n",
                "#     return out\n",
                "\n",
                "# train_markup = build_split(train_dataset)\n",
                "# val_markup   = build_split(val_dataset)\n",
                "# test_markup  = build_split(test_dataset)"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 21,
            "id": "70f01bf6",
            "metadata": {},
            "outputs": [],
            "source": [
                "# def return_doc_markup(i, page_num):\n",
                "#     x = DocumentSerializer.serialize_document(i)['pages'][page_num]\n",
                "#     x[\"bounding_boxes\"] = x[\"total_bboxes\"]\n",
                "#     x[\"image_path\"] = x[\"metadata\"][\"image_path\"]\n",
                "#     x[\"classification_data\"] = DocumentSerializer.serialize_document(i)['document_classes']\n",
                "#     return x\n",
                "\n",
                "\n",
                "# train_markup, val_markup, test_markup = [], [], []\n",
                "# for i in tqdm.tqdm(train_dataset):\n",
                "#     try:\n",
                "#         for pn, page in enumerate(i.pages):\n",
                "#             train_markup.append(return_doc_markup(i, pn))\n",
                "#     except:\n",
                "#         print(f\"Problem with {i}\")\n",
                "\n",
                "# for i in val_dataset:\n",
                "#     try:\n",
                "#         for pn, page in enumerate(i.pages):\n",
                "#             val_markup.append(return_doc_markup(i, pn))\n",
                "#     except:\n",
                "#         print(f\"Problem with {i}\")\n",
                "\n",
                "# for i in test_dataset:\n",
                "#     try:\n",
                "#         for pn, page in enumerate(i.pages):\n",
                "#             test_markup.append(return_doc_markup(i, pn))\n",
                "#     except:\n",
                "#         print(f\"Problem with {i}\")"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 22,
            "id": "6732e58c",
            "metadata": {},
            "outputs": [],
            "source": [
                "# test_markup[0]"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 23,
            "id": "41c79208",
            "metadata": {},
            "outputs": [
                {
                    "name": "stdout",
                    "output_type": "stream",
                    "text": [
                        "Number of documents in train dataset: 8318\n",
                        "Number of documents of val dataset: 1373\n",
                        "Number of documents of test dataset: 2122\n"
                    ]
                }
            ],
            "source": [
                "train_dataset_size = len(train_dataset)\n",
                "val_dataset_size = len(val_dataset)\n",
                "test_dataset_size = len(test_dataset)\n",
                "print(f\"Number of documents in train dataset: {len(train_dataset)}\")\n",
                "print(f\"Number of documents of val dataset: {len(val_dataset)}\")\n",
                "print(f\"Number of documents of test dataset: {len(test_dataset)}\")"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 24,
            "id": "77d67228",
            "metadata": {},
            "outputs": [
                {
                    "name": "stderr",
                    "output_type": "stream",
                    "text": [
                        "/home/iana/apron-william/apron_env/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().\n",
                        "  warnings.warn(_BETA_TRANSFORMS_WARNING)\n",
                        "/home/iana/apron-william/apron_env/lib/python3.10/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().\n",
                        "  warnings.warn(_BETA_TRANSFORMS_WARNING)\n",
                        "/home/iana/apron-william/apron_env/lib/python3.10/site-packages/transformers/models/layoutlmv3/feature_extraction_layoutlmv3.py:30: FutureWarning: The class LayoutLMv3FeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use LayoutLMv3ImageProcessor instead.\n",
                        "  warnings.warn(\n"
                    ]
                },
                {
                    "name": "stdout",
                    "output_type": "stream",
                    "text": [
                        "Number of pages in train dataset: 11348\n",
                        "Number of pages of val dataset: 1737\n",
                        "Number of pages of test dataset: 2693\n"
                    ]
                }
            ],
            "source": [
                "# lilt model\n",
                "from transformers import LayoutLMv3FeatureExtractor, AutoTokenizer, LayoutLMv3Processor\n",
                "\n",
                "feature_extractor = LayoutLMv3FeatureExtractor(apply_ocr=False)\n",
                "tokenizer = AutoTokenizer.from_pretrained(HUGGINGFACE_MODEL_ID)\n",
                "# cannot use from_pretrained since the processor is not saved in the base model\n",
                "processor = LayoutLMv3Processor(feature_extractor, tokenizer)\n",
                "\n",
                "from model_training.page_augmentator import PageAugmentator\n",
                "\n",
                "augmentator = PageAugmentator(\n",
                "    seed=42,\n",
                "    words_removal_proba=0.1,\n",
                "    max_words_to_remove_share=0.1,\n",
                "    words_shuffle_proba=0.4,\n",
                "    swap_proba=0.4,\n",
                "    neighbours_proba=0.8,\n",
                "    neighbours_window=5,\n",
                "    geometry_augmentation_proba=0.8,\n",
                "    page_shift_proba=0.8,\n",
                "    max_page_shift=0.4,\n",
                "    bboxes_shift_proba=1,\n",
                "    bbox_shift_proba=0.5,\n",
                "    page_rotation_proba=0.2,\n",
                "    max_page_rotation_angle=20,\n",
                "    bboxes_rotation_proba=0.6,\n",
                "    bbox_rotation_proba=0.3,\n",
                "    max_bbox_rotation_angle=15,\n",
                ")\n",
                "\n",
                "# from william.inference.model_configs.invoices_model import BASIC_INVOICES_MODEL_CONFIG\n",
                "from william.inference.model_configs.invoices_model import (\n",
                "    ROW_NUMBERS_PREDICTION_NAME,\n",
                "    get_invoices_model_config,\n",
                ")\n",
                "\n",
                "# config for labelled_train__lr_4e-5__good_dataset__a100 and labelled_train__lr_4e-5__good_dataset/epoch=50.ckpt\n",
                "model_config = get_invoices_model_config(\n",
                "        \"ground_truth\",\n",
                "        \"ground_truth_binary\",\n",
                "        \"gt_line_items_rows\",\n",
                "        \"invofox_line_items\",\n",
                "        ROW_NUMBERS_PREDICTION_NAME,\n",
                "    )\n",
                "\n",
                "train_dataset = WilliamModelDataset(\n",
                "    train_markup,\n",
                "    processor,\n",
                "    DATASET_PATH,\n",
                "    model_config,\n",
                "    augmentator,\n",
                "    \"first_tokens\",\n",
                ")\n",
                "val_dataset = WilliamModelDataset(\n",
                "    val_markup,\n",
                "    processor,\n",
                "    DATASET_PATH,\n",
                "    model_config,\n",
                "    None,\n",
                ")\n",
                "test_dataset = WilliamModelDataset(\n",
                "    test_markup,\n",
                "    processor,\n",
                "    DATASET_PATH,\n",
                "    model_config,\n",
                "    None,\n",
                ")\n",
                "# test_no_bbox_dataset = InvoiceTrainingDataset(test_no_bbox_markup, processor, DATASET_PATH, GROUND_TRUTH_MARKUP)\n",
                "\n",
                "print(f\"Number of pages in train dataset: {len(train_dataset)}\")\n",
                "print(f\"Number of pages of val dataset: {len(val_dataset)}\")\n",
                "print(f\"Number of pages of test dataset: {len(test_dataset)}\")"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "id": "e8208a57",
            "metadata": {},
            "outputs": [],
            "source": []
        },
        {
            "cell_type": "markdown",
            "id": "8762d98e",
            "metadata": {},
            "source": [
                "#### Some checks"
            ]
        },
        {
            "attachments": {},
            "cell_type": "markdown",
            "id": "52339581",
            "metadata": {
                "id": "52339581"
            },
            "source": [
                "## Training with pytorch lightning"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 25,
            "id": "HfKkFeTMG_59",
            "metadata": {
                "id": "HfKkFeTMG_59"
            },
            "outputs": [],
            "source": [
                "import torch\n",
                "from torch.utils.data import DataLoader\n",
                "import lightning.pytorch as pl\n",
                "from lightning.pytorch.callbacks import ModelCheckpoint\n",
                "from lightning.pytorch.callbacks.early_stopping import EarlyStopping\n",
                "from lightning.pytorch.callbacks import TQDMProgressBar\n",
                "from lightning.pytorch.loggers import TensorBoardLogger, MLFlowLogger"
            ]
        },
        {
            "attachments": {},
            "cell_type": "markdown",
            "id": "oLAVlDyBNtzr",
            "metadata": {
                "id": "oLAVlDyBNtzr"
            },
            "source": [
                "#### Trainer and DataLoader"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 26,
            "id": "5810bab8",
            "metadata": {},
            "outputs": [],
            "source": [
                "CHECKPOINT_TO_RESUME_FROM = None\n",
                "# CHECKPOINT_TO_RESUME_FROM = \"/content/drive/MyDrive/apron/model_checkpoints/labelled_train_v2__lr_4e-5/epoch=35-step=8964-val_loss=0.2748.ckpt\""
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 27,
            "id": "ea7c19f2",
            "metadata": {},
            "outputs": [
                {
                    "data": {
                        "text/plain": [
                            "'capture-2216-train-new-dataset'"
                        ]
                    },
                    "execution_count": 27,
                    "metadata": {},
                    "output_type": "execute_result"
                }
            ],
            "source": [
                "EXPERIMENT_NAME"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 28,
            "id": "17a50cec",
            "metadata": {
                "id": "17a50cec"
            },
            "outputs": [
                {
                    "name": "stdout",
                    "output_type": "stream",
                    "text": [
                        "Batch size: 12\n"
                    ]
                },
                {
                    "name": "stderr",
                    "output_type": "stream",
                    "text": [
                        "GPU available: True (cuda), used: True\n",
                        "TPU available: False, using: 0 TPU cores\n",
                        "IPU available: False, using: 0 IPUs\n",
                        "HPU available: False, using: 0 HPUs\n"
                    ]
                }
            ],
            "source": [
                "batch_size = 32\n",
                "if torch.cuda.is_available():\n",
                "    if torch.cuda.get_device_name().startswith(\"Tesla V100\"):\n",
                "        batch_size = 12  # took 14.8 GB GPU memory\n",
                "    elif torch.cuda.get_device_name().startswith(\"NVIDIA L4\"):\n",
                "        batch_size = 12\n",
                "    else:  # A100:\n",
                "        batch_size = 32\n",
                "print(\"Batch size:\", batch_size)\n",
                "\n",
                "\n",
                "def seed_worker(worker_id):\n",
                "    worker_seed = torch.initial_seed() % 2**32\n",
                "    np.random.seed(worker_seed)\n",
                "    random.seed(worker_seed)\n",
                "\n",
                "\n",
                "generator_train = torch.Generator()\n",
                "generator_train.manual_seed(0)\n",
                "train_data_loader = DataLoader(\n",
                "    train_dataset,\n",
                "    batch_size=batch_size,\n",
                "    shuffle=True,\n",
                "    num_workers=2 if torch.cuda.is_available() else 0,\n",
                "    worker_init_fn=seed_worker,\n",
                "    generator=generator_train,\n",
                ")\n",
                "\n",
                "generator_val = torch.Generator()\n",
                "generator_val.manual_seed(0)\n",
                "val_data_loader = DataLoader(\n",
                "    val_dataset,\n",
                "    batch_size=batch_size,\n",
                "    shuffle=False,\n",
                "    num_workers=2 if torch.cuda.is_available() else 0,\n",
                "    worker_init_fn=seed_worker,\n",
                "    generator=generator_val,\n",
                ")\n",
                "\n",
                "model_checkpoint = ModelCheckpoint(\n",
                "    filename=\"{epoch}-{step}-{val_loss:.4f}\",\n",
                "    save_last=True,\n",
                "    save_top_k=-1,\n",
                "    every_n_epochs=1,\n",
                "    monitor=\"val_loss\",\n",
                "    mode=\"min\",\n",
                "    dirpath=EXPERIMENT_CHECKPOINTS_PATH,  # ,\n",
                "    # every_n_train_steps=100\n",
                ")\n",
                "early_stopping = EarlyStopping(monitor=\"val_loss\", mode=\"min\", patience=100, verbose=True)\n",
                "\n",
                "training_params = {\n",
                "    \"tokens_weight_decay\": 0.0001,\n",
                "    \"doc_weight_decay\": 0.001,\n",
                "    \"reduce_lr_on_plateau_factor\": 0.5,\n",
                "    \"reduce_lr_on_plateau_patience\": 25,  # was 10\n",
                "    \"backbone_weight_decay\": 1e-2,\n",
                "}\n",
                "\n",
                "hyperparams = {\n",
                "    \"dataset\": \"buyer_info\",\n",
                "    \"model\": HUGGINGFACE_MODEL_ID,\n",
                "    \"learning_rate\": LEARNING_RATE,\n",
                "    \"train_size\": train_dataset_size, #len(train_markup_in_files), # UNDERSTAND IF NEEDED\n",
                "    \"val_size\": val_dataset_size, #len(val_markup_in_files), # UNDERSTAND IF NEEDED\n",
                "    \"test_size\": test_dataset_size, #len(test_markup_in_files), # UNDERSTAND IF NEEDED\n",
                "    \"train_size_in_pages\": len(train_dataset),\n",
                "    \"val_size_in_pages\": len(val_dataset),\n",
                "    \"test_size_in_pages\": len(test_dataset),\n",
                "    **training_params,\n",
                "}\n",
                "tb_logger = TensorBoardLogger(TENSORBOARD_DIR, name=EXPERIMENT_NAME, log_graph=False)\n",
                "tb_logger.log_hyperparams(hyperparams)\n",
                "mlf_logger = MLFlowLogger(\n",
                "    experiment_name=EXPERIMENT_NAME,\n",
                "    tracking_uri=MLFLOW_TRACKING_URI,\n",
                "    log_model=False,\n",
                "    run_name=f\"{EXPERIMENT_NAME}_{RUN_NAME}\",\n",
                ")\n",
                "mlf_logger.log_hyperparams(hyperparams)\n",
                "\n",
                "trainer = pl.Trainer(\n",
                "    accelerator=\"gpu\", # \"cpu\",\n",
                "    # limit_train_batches=1,\n",
                "    # limit_val_batches=1,\n",
                "    # precision=32,\n",
                "    devices=1,\n",
                "    max_epochs=50, # 1000,\n",
                "    # callbacks=[model_checkpoint, early_stopping, TQDMProgressBar(refresh_rate=1)],\n",
                "    callbacks=[model_checkpoint, TQDMProgressBar(refresh_rate=1)],\n",
                "    num_sanity_val_steps=0,\n",
                "    logger=[tb_logger, mlf_logger],\n",
                "    log_every_n_steps=1,\n",
                ")"
            ]
        },
        {
            "cell_type": "markdown",
            "id": "28db48e7",
            "metadata": {},
            "source": [
                "#### Mean and std evaluation for RegressionRowsLabelEncoder"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 29,
            "id": "b40dbcab",
            "metadata": {},
            "outputs": [],
            "source": [
                "# from william.inference.model_multitask import RegressionRowsLabelEncoder\n",
                "#\n",
                "# encoder = RegressionRowsLabelEncoder('test')#, mean=3.113227, std=3.186576)\n",
                "#\n",
                "# all_values = []\n",
                "# for doc in train_dataset + val_dataset:\n",
                "#     encoded = encoder.encode(doc['line_items_rows_labels'])\n",
                "#     values = encoded[encoded != -100]\n",
                "#     all_values.extend(values)\n",
                "#\n",
                "# mean = np.mean(all_values)\n",
                "# std = np.std(all_values, ddof=0)\n",
                "#\n",
                "# mean, std"
            ]
        },
        {
            "cell_type": "markdown",
            "id": "45d0463b",
            "metadata": {},
            "source": [
                "#### Document classification model training"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 30,
            "id": "1a267156",
            "metadata": {},
            "outputs": [
                {
                    "name": "stderr",
                    "output_type": "stream",
                    "text": [
                        "Some weights of LiltForTokenClassification were not initialized from the model checkpoint at SCUT-DLVCLab/lilt-roberta-en-base and are newly initialized: ['classifier.bias', 'classifier.weight']\n",
                        "You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.\n"
                    ]
                }
            ],
            "source": [
                "from invoices_dataset.constants import DOCUMENT_CLASSES\n",
                "from william.inference.model_multitask import MutltiTaskModule\n",
                "\n",
                "model_module = MutltiTaskModule(\n",
                "    model_config=model_config,\n",
                "    model_doc_classes=DOCUMENT_CLASSES, lr=LEARNING_RATE, **training_params\n",
                ")"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 31,
            "id": "7f12e243",
            "metadata": {},
            "outputs": [
                {
                    "name": "stderr",
                    "output_type": "stream",
                    "text": [
                        "It looks like you are trying to rescale already rescaled images. If the input images have pixel values between 0 and 1, set `do_rescale=False` to avoid rescaling them again.\n",
                        "You're using a LayoutLMv3TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.\n",
                        "It looks like you are trying to rescale already rescaled images. If the input images have pixel values between 0 and 1, set `do_rescale=False` to avoid rescaling them again.\n",
                        "You're using a LayoutLMv3TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.\n",
                        "/home/iana/apron-william/apron_env/lib/python3.10/site-packages/lightning/pytorch/core/module.py:423: You are trying to `self.log()` but the `self.trainer` reference is not registered on the model yet. This is most likely because the model hasn't been passed to the `Trainer`\n"
                    ]
                },
                {
                    "data": {
                        "text/plain": [
                            "tensor(2.6450, grad_fn=<DivBackward0>)"
                        ]
                    },
                    "execution_count": 31,
                    "metadata": {},
                    "output_type": "execute_result"
                }
            ],
            "source": [
                "batch = next(iter(train_data_loader))\n",
                "model_module.training_step(batch, None)"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 32,
            "id": "d0e88a7d",
            "metadata": {},
            "outputs": [],
            "source": [
                "# Debug\n",
                "# batch1 = next(iter(val_data_loader))\n",
                "# model_module.training_step(batch1, None)"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 33,
            "id": "475e2350",
            "metadata": {},
            "outputs": [],
            "source": [
                "CHECKPOINT_TO_RESUME_FROM"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 34,
            "id": "3ea4ac45",
            "metadata": {},
            "outputs": [
                {
                    "name": "stderr",
                    "output_type": "stream",
                    "text": [
                        "You are using a CUDA device ('NVIDIA L4') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision\n",
                        "/home/iana/apron-william/apron_env/lib/python3.10/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:639: Checkpoint directory /home/iana/training/model_checkpoints/capture-2216-train-new-dataset exists and is not empty.\n",
                        "LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]\n",
                        "\n",
                        "  | Name                 | Type                       | Params\n",
                        "--------------------------------------------------------------------\n",
                        "0 | _tokens_model        | LiltForTokenClassification | 130 M \n",
                        "1 | _sequence_classifier | Linear                     | 10.0 K\n",
                        "--------------------------------------------------------------------\n",
                        "130 M     Trainable params\n",
                        "0         Non-trainable params\n",
                        "130 M     Total params\n",
                        "521.116   Total estimated model params size (MB)\n"
                    ]
                },
                {
                    "name": "stdout",
                    "output_type": "stream",
                    "text": [
                        "Epoch 0:   0%|          | 0/946 [00:00<?, ?it/s] "
                    ]
                },
                {
                    "name": "stderr",
                    "output_type": "stream",
                    "text": [
                        "It looks like you are trying to rescale already rescaled images. If the input images have pixel values between 0 and 1, set `do_rescale=False` to avoid rescaling them again.\n",
                        "You're using a LayoutLMv3TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.\n",
                        "It looks like you are trying to rescale already rescaled images. If the input images have pixel values between 0 and 1, set `do_rescale=False` to avoid rescaling them again.\n",
                        "You're using a LayoutLMv3TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.\n"
                    ]
                },
                {
                    "name": "stdout",
                    "output_type": "stream",
                    "text": [
                        "Epoch 0: 100%|██████████| 946/946 [49:38<00:00,  0.32it/s, v_num=9074]"
                    ]
                },
                {
                    "name": "stderr",
                    "output_type": "stream",
                    "text": [
                        "It looks like you are trying to rescale already rescaled images. If the input images have pixel values between 0 and 1, set `do_rescale=False` to avoid rescaling them again.\n",
                        "It looks like you are trying to rescale already rescaled images. If the input images have pixel values between 0 and 1, set `do_rescale=False` to avoid rescaling them again.\n",
                        "You're using a LayoutLMv3TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.\n",
                        "You're using a LayoutLMv3TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.\n"
                    ]
                },
                {
                    "name": "stdout",
                    "output_type": "stream",
                    "text": [
                        "Epoch 1:   0%|          | 0/946 [00:00<?, ?it/s, v_num=9074, val_loss=nan.0]          "
                    ]
                },
                {
                    "name": "stderr",
                    "output_type": "stream",
                    "text": [
                        "It looks like you are trying to rescale already rescaled images. If the input images have pixel values between 0 and 1, set `do_rescale=False` to avoid rescaling them again.\n",
                        "You're using a LayoutLMv3TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.\n",
                        "It looks like you are trying to rescale already rescaled images. If the input images have pixel values between 0 and 1, set `do_rescale=False` to avoid rescaling them again.\n",
                        "You're using a LayoutLMv3TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.\n"
                    ]
                },
                {
                    "name": "stdout",
                    "output_type": "stream",
                    "text": [
                        "Epoch 1:  96%|█████████▌| 909/946 [48:15<01:57,  0.31it/s, v_num=9074, val_loss=nan.0]"
                    ]
                }
            ],
            "source": [
                "trainer.fit(\n",
                "    model_module,\n",
                "    train_data_loader,\n",
                "    val_data_loader,\n",
                "    ckpt_path=CHECKPOINT_TO_RESUME_FROM,\n",
                ")"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 1,
            "id": "62efdf17",
            "metadata": {},
            "outputs": [
                {
                    "data": {
                        "text/plain": [
                            "4"
                        ]
                    },
                    "execution_count": 1,
                    "metadata": {},
                    "output_type": "execute_result"
                }
            ],
            "source": [
                "2+2"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "id": "58541047",
            "metadata": {},
            "outputs": [],
            "source": []
        },
        {
            "attachments": {},
            "cell_type": "markdown",
            "id": "DLqqNyHt_Zt8",
            "metadata": {
                "id": "DLqqNyHt_Zt8"
            },
            "source": [
                "## Tensorboard"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 32,
            "id": "mBFIbA0ZHjPY",
            "metadata": {
                "id": "mBFIbA0ZHjPY"
            },
            "outputs": [],
            "source": [
                "!kill $(ps -e | grep 'tensorboard' | awk '{print $1}')\n",
                "#!pip install -qqq tensorboardX==2.5.1\n",
                "#!ls lightning_logs/version_7/checkpoints/"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 25,
            "id": "016315d8",
            "metadata": {},
            "outputs": [
                {
                    "name": "stdout",
                    "output_type": "stream",
                    "text": [
                        "version_0   version_23\tversion_38  version_52\tversion_67  version_81\n",
                        "version_1   version_24\tversion_39  version_53\tversion_68  version_82\n",
                        "version_10  version_25\tversion_4   version_54\tversion_69  version_83\n",
                        "version_11  version_26\tversion_40  version_55\tversion_7   version_84\n",
                        "version_12  version_27\tversion_41  version_56\tversion_70  version_85\n",
                        "version_13  version_28\tversion_42  version_57\tversion_71  version_86\n",
                        "version_14  version_29\tversion_43  version_58\tversion_72  version_87\n",
                        "version_15  version_3\tversion_44  version_59\tversion_73  version_88\n",
                        "version_16  version_30\tversion_45  version_6\tversion_74  version_89\n",
                        "version_17  version_31\tversion_46  version_60\tversion_75  version_9\n",
                        "version_18  version_32\tversion_47  version_61\tversion_76  version_90\n",
                        "version_19  version_33\tversion_48  version_62\tversion_77  version_91\n",
                        "version_2   version_34\tversion_49  version_63\tversion_78\n",
                        "version_20  version_35\tversion_5   version_64\tversion_79\n",
                        "version_21  version_36\tversion_50  version_65\tversion_8\n",
                        "version_22  version_37\tversion_51  version_66\tversion_80\n"
                    ]
                }
            ],
            "source": [
                "! ls /home/iana/training/tb/capture-2216-train-new-dataset/"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 26,
            "id": "IHx6g9veGkLv",
            "metadata": {
                "id": "IHx6g9veGkLv"
            },
            "outputs": [
                {
                    "data": {
                        "text/plain": [
                            "Launching TensorBoard..."
                        ]
                    },
                    "metadata": {},
                    "output_type": "display_data"
                }
            ],
            "source": [
                "%load_ext tensorboard\n",
                "#%reload_ext tensorboard\n",
                "#%tensorboard --logdir '/home/jupyter/gcs/tensorboard_logs'\n",
                "%tensorboard --logdir '/home/iana/training/tb/capture-2216-train-new-dataset/version_91'"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": 36,
            "id": "5ce4ba6f",
            "metadata": {},
            "outputs": [],
            "source": [
                "# For runnging tensorboard locally:\n",
                "# tensorboard --port 6007 --logdir ../apron_drive/training/tensorboard_logs/\n",
                "# tensorboard --port 6006 --logdir /home/iana/training/tb/capture-2216-train-new-dataset/version_85"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "id": "4e13674e",
            "metadata": {},
            "outputs": [],
            "source": []
        }
    ],
    "metadata": {
        "accelerator": "GPU",
        "colab": {
            "collapsed_sections": [
                "b2984d0a"
            ],
            "gpuType": "V100",
            "provenance": []
        },
        "gpuClass": "premium",
        "kernelspec": {
            "display_name": "apron-william",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "codemirror_mode": {
                "name": "ipython",
                "version": 3
            },
            "file_extension": ".py",
            "mimetype": "text/x-python",
            "name": "python",
            "nbconvert_exporter": "python",
            "pygments_lexer": "ipython3",
            "version": "3.10.15"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 5
}
